# 05 — Entrenamiento de Producción

Lee los mejores parámetros directamente desde MLflow (runs del `04_grid_search`),
entrena el modelo TwoStage sobre **todos los datos disponibles** y lo sube a Hugging Face Hub.

**Fuente de parámetros:**
- Mejor clf → run con `tags.stage = 'clf_grid'`, ordenado por `avg_fbeta` DESC
- Mejor reg → run con `tags.stage = 'reg_grid'` y mismo `tags.clf_fixed`, ordenado por `avg_rmse_rain` ASC
- Threshold y métricas baseline → run `tags.stage = 'two_stage_final'`

**Outputs:** `two_stage_model.pkl` + `metadata.json` en Hugging Face Hub

In [0]:
%pip install huggingface_hub lightgbm joblib --quiet
dbutils.library.restartPython()


In [0]:
import os, json, joblib, mlflow, numpy as np, pandas as pd
from datetime import datetime, timezone
from src.utils.config import EXPERIMENT_NAME

# ── Hugging Face ────────────────────────────────────────────────────
# Opción A (recomendada): guardar token en Databricks Secrets
#   1. En Databricks: Settings → Secrets → Create scope 'hf'
#   2. databricks secrets put --scope hf --key token
# HF_TOKEN    = dbutils.secrets.get(scope="hf", key="HF_TOKEN")
# Opción B (directa, solo para desarrollo):
HF_TOKEN = os.getenv("HF_TOKEN", "")

HF_USERNAME = "desareca"   # <── cambiar por tu usuario de HF
HF_REPO_ID  = f"{HF_USERNAME}/santiago-weather"

MODEL_DIR  = "/tmp/santiago_weather_model"
MODEL_PATH = f"{MODEL_DIR}/two_stage_model.pkl"
META_PATH  = f"{MODEL_DIR}/metadata.json"
os.makedirs(MODEL_DIR, exist_ok=True)

mlflow.set_experiment(EXPERIMENT_NAME)
EXPERIMENT_ID = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

print(f'✅ Config lista | Experimento: {EXPERIMENT_NAME} | HF: {HF_REPO_ID}')

In [0]:
def extract_params(run: pd.Series, prefix: str = '') -> dict:
    """
    Extrae parámetros de una fila de search_runs(), casteando al tipo correcto.
    MLflow guarda todo como string — hay que convertir int/float/bool.
    """
    def cast(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return None
        v = str(v).strip()
        if v.lower() in ('true', 'false'):
            return v.lower() == 'true'
        # Intentar int solo si no tiene punto decimal
        if '.' not in v:
            try: return int(v)
            except ValueError: pass
        try: return float(v)
        except ValueError: pass
        return v

    result = {}
    for col in run.index:
        if col.startswith(f'params.{prefix}'):
            key = col[len(f'params.{prefix}'):]
            result[key] = cast(run[col])
    return {k: v for k, v in result.items() if v is not None}

# ── 2.1 Mejor clasificador ──────────────────────────────────────────
# NOTA: el reg_grid se corrió con tags.clf_fixed = 'best_clf' (literal),
# y el two_stage_final tiene el clf real en tags.clf_name.
# Por eso leemos el clf desde two_stage_final, no desde clf_grid directamente.

# Primero leemos el run final — es la fuente de verdad de la combinación ganadora
final_runs = mlflow.search_runs(
    experiment_ids=[EXPERIMENT_ID],
    filter_string="tags.stage = 'two_stage_final'",
    order_by=["metrics.test_rmse ASC"],
)
assert len(final_runs) > 0, '❌ Sin runs two_stage_final. Corre primero el notebook 04 completo.'

best_final_run = final_runs.iloc[0]
BEST_CLF_NAME  = best_final_run.get('tags.clf_name', 'desconocido')
BEST_REG_NAME  = best_final_run.get('tags.reg_name', 'desconocido')

print(f'📋 Combinación ganadora (desde two_stage_final):')
print(f'   CLF: {BEST_CLF_NAME}')
print(f'   REG: {BEST_REG_NAME}')
print(f'   test_rmse:   {best_final_run.get("metrics.test_rmse", float("nan")):.3f}')
print(f'   test_recall: {best_final_run.get("metrics.test_recall", float("nan")):.3f}')


# ── 2.2 Params del clasificador ─────────────────────────────────────
clf_runs = mlflow.search_runs(
    experiment_ids=[EXPERIMENT_ID],
    filter_string=f"tags.stage = 'clf_grid' and tags.config_name = '{BEST_CLF_NAME}'",
)
assert len(clf_runs) > 0, f'❌ No se encontró run clf_grid para {BEST_CLF_NAME}.'

best_clf_run    = clf_runs.iloc[0]
_all_clf_params = extract_params(best_clf_run)
_no_model_keys  = {'threshold', 'log_target', 'clf_rain_threshold', 'reg_rain_threshold', 'n_splits'}
BEST_CLF_PARAMS = {k: v for k, v in _all_clf_params.items() if k not in _no_model_keys}

print(f'\n🏆 CLF params: {BEST_CLF_PARAMS}')


# ── 2.3 Params del regresor ─────────────────────────────────────────
reg_runs = mlflow.search_runs(
    experiment_ids=[EXPERIMENT_ID],
    filter_string=f"tags.stage = 'reg_grid' and tags.config_name = '{BEST_REG_NAME}'",
)
assert len(reg_runs) > 0, f'❌ No se encontró run reg_grid para {BEST_REG_NAME}.'

best_reg_run    = reg_runs.iloc[0]
_all_reg_params = extract_params(best_reg_run)
BEST_REG_PARAMS = {k: v for k, v in _all_reg_params.items() if k not in _no_model_keys}

print(f'🏆 REG params: {BEST_REG_PARAMS}')


# ── 2.4 Threshold y métricas baseline desde el run final ────────────
BEST_CLF_THRESHOLD = float(best_final_run.get('params.threshold', 0.3))
BEST_LOG_TARGET    = str(best_final_run.get('params.log_target', 'True')).lower() == 'true'

BASELINE_METRICS = {
    k: float(best_final_run.get(f'metrics.test_{k}', np.nan))
    for k in ['rmse', 'r2', 'recall', 'recall_picos', 'auc', 'mae_rain', 'rmse_rain']
}
BASELINE_METRICS = {k: v for k, v in BASELINE_METRICS.items() if not np.isnan(v)}

print(f'\n📋 Threshold: {BEST_CLF_THRESHOLD} | log_target: {BEST_LOG_TARGET}')
print(f'📊 Baseline metrics: {BASELINE_METRICS}')


In [0]:
from src.data.ingestion import load_from_delta_table
from src.data.preprocessing import build_features

df_raw  = load_from_delta_table('weather_raw', spark)
df_full = build_features(df_raw).dropna()

assert df_full.isna().sum().sum() == 0, '❌ NaN encontrados'
print(f'📊 Período: {df_full.index.min().date()} → {df_full.index.max().date()} | {len(df_full)} días')

In [0]:
from src.models.two_stage_model import TwoStagePredictor
from src.evaluation.two_stage_cv import evaluate_two_stage

df_val_prod  = df_full.iloc[-365:]
df_train_val = df_full.iloc[:-365]

model_val = TwoStagePredictor(
    clf_params=BEST_CLF_PARAMS, reg_params=BEST_REG_PARAMS,
    threshold=BEST_CLF_THRESHOLD, log_target=BEST_LOG_TARGET,
)
model_val.fit(df_train_val)

val_metrics = evaluate_two_stage(
    y_true=df_val_prod['precip_t1'],
    y_pred=model_val.predict(df_val_prod),
    prob_llueve=model_val.predict_proba(df_val_prod),
    threshold=BEST_CLF_THRESHOLD,
    label=f'Pre-prod ({df_val_prod.index.min().date()} → {df_val_prod.index.max().date()})',
    verbose=True,
)

print(f'\n📊 Comparativa con baseline test holdout:')
print(f'   {"Métrica":20s}  {"Baseline":>10}  {"Val actual":>12}  {"Δ":>8}')
print(f'   {"-"*54}')
for k, b_val in BASELINE_METRICS.items():
    curr = val_metrics.get(k, np.nan)
    if not np.isnan(curr):
        print(f'   {k:20s}  {b_val:>10.3f}  {curr:>12.3f}  {curr - b_val:>+8.3f}')

In [0]:
print(f'🚀 Entrenando sobre todos los datos ({len(df_full)} días)...')

model_prod = TwoStagePredictor(
    clf_params=BEST_CLF_PARAMS, reg_params=BEST_REG_PARAMS,
    threshold=BEST_CLF_THRESHOLD, log_target=BEST_LOG_TARGET,
)
model_prod.fit(df_full)
print(f'✅ Entrenado | {len(model_prod.feature_names)} features')

In [0]:
joblib.dump(model_prod, MODEL_PATH, compress=3)
model_size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
version       = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')

metadata = {
    'version':           version,
    'trained_at':        datetime.now(timezone.utc).isoformat(),
    'train_start':       str(df_full.index.min().date()),
    'train_end':         str(df_full.index.max().date()),
    'n_samples':         len(df_full),
    'n_features':        len(model_prod.feature_names),
    'feature_names':     model_prod.feature_names,
    # Trazabilidad — qué runs de MLflow originaron este modelo
    'mlflow_experiment': EXPERIMENT_NAME,
    'clf_run_id':        best_clf_run['run_id'],
    'reg_run_id':        best_reg_run['run_id'],
    'final_run_id':      best_final_run['run_id'],
    'clf_name':          BEST_CLF_NAME,
    'reg_name':          BEST_REG_NAME,
    # Arquitectura
    'model_type':           'TwoStagePredictor',
    'clf_params':           BEST_CLF_PARAMS,
    'reg_params':           BEST_REG_PARAMS,
    'threshold':            BEST_CLF_THRESHOLD,
    'log_target':           BEST_LOG_TARGET,
    'clf_rain_threshold':   0.5,
    'reg_rain_threshold':   0.5,
    # Métricas del test holdout original — referencia para detectar degradación
    'baseline_metrics':     BASELINE_METRICS,
    # Umbrales que disparan reentrenamiento automático desde el monthly_flow
    # Se activa si AMBAS condiciones se cumplen en los últimos 30 días
    'degradation_thresholds': {
        'rmse_max_pct_increase': 0.20,  # RMSE no puede subir más del 20%
        'recall_min':            0.50,  # Recall lluvia no puede bajar de 50%
    },
    'hf_repo':           HF_REPO_ID,
    'model_filename':    'two_stage_model.pkl',
}

with open(META_PATH, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f'✅ Serializado: {model_size_mb:.2f} MB | Versión: {version}')

In [0]:
from huggingface_hub import HfApi, create_repo

api = HfApi(token=HF_TOKEN)
create_repo(repo_id=HF_REPO_ID, token=HF_TOKEN, repo_type='model', private=False, exist_ok=True)

commit_msg = f'Model {version} | clf={BEST_CLF_NAME} reg={BEST_REG_NAME} thr={BEST_CLF_THRESHOLD}'

for local_path, remote_name in [(MODEL_PATH, 'two_stage_model.pkl'), (META_PATH, 'metadata.json')]:
    api.upload_file(
        path_or_fileobj=local_path, path_in_repo=remote_name,
        repo_id=HF_REPO_ID, repo_type='model', commit_message=commit_msg,
    )
    print(f'  ✅ {remote_name}')

print(f'\n🎉 https://huggingface.co/{HF_REPO_ID}')

In [0]:
from huggingface_hub import hf_hub_download

dl_model  = hf_hub_download(repo_id=HF_REPO_ID, filename='two_stage_model.pkl', cache_dir='/tmp/hf_verify')
dl_meta   = hf_hub_download(repo_id=HF_REPO_ID, filename='metadata.json',       cache_dir='/tmp/hf_verify')
m_verify  = joblib.load(dl_model)
with open(dl_meta) as f: meta_v = json.load(f)

last_row  = df_full.iloc[[-1]]
test_pred = m_verify.predict(last_row)
test_prob = m_verify.predict_proba(last_row)

print(f'✅ Verificación exitosa')
print(f'   Versión: {meta_v["version"]} | Features: {len(m_verify.feature_names)}')
print(f'   Predicción para {df_full.index[-1].date()} + 1:')
print(f'     P(llueve): {test_prob[0]:.3f}')
print(f'     mm pred:   {test_pred.iloc[0]:.2f} mm')
print(f'\n🚀 Listo para Render. Variables de entorno a configurar:')
print(f'     HF_REPO_ID = {HF_REPO_ID}')
print(f'     HF_TOKEN   = <token con permisos de lectura>')

In [0]:
with mlflow.start_run(run_name=f'production_{version}') as run:
    mlflow.set_tag('run_type',      'production')
    mlflow.set_tag('stage',         'production')
    mlflow.set_tag('model_version', version)
    mlflow.set_tag('hf_repo',       HF_REPO_ID)
    mlflow.set_tag('clf_name',      BEST_CLF_NAME)
    mlflow.set_tag('reg_name',      BEST_REG_NAME)
    mlflow.set_tag('clf_run_id',    best_clf_run['run_id'])
    mlflow.set_tag('reg_run_id',    best_reg_run['run_id'])
    mlflow.set_tag('final_run_id',  best_final_run['run_id'])
    mlflow.log_param('threshold',   BEST_CLF_THRESHOLD)
    mlflow.log_param('log_target',  BEST_LOG_TARGET)
    mlflow.log_param('n_samples',   len(df_full))
    mlflow.log_param('train_end',   str(df_full.index.max().date()))
    for k, v in BASELINE_METRICS.items():
        mlflow.log_metric(f'baseline_{k}', v)
    for k, v in val_metrics.items():
        try:
            if not np.isnan(float(v)):
                mlflow.log_metric(f'val_prod_{k}', float(v))
        except (TypeError, ValueError):
            pass
    mlflow.log_artifact(META_PATH)

print(f'✅ Run de producción loggeado: {run.info.run_id}')